Імпорти

In [8]:
import sys
import time
import joblib
from pathlib import Path

import pandas as pd

from sklearn.dummy import DummyClassifier

from src.config import MODEL_RESULTS_PATH
from src.evaluation import save_experiment_result
from sklearn.model_selection import GridSearchCV

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import (
    TRAIN_PATH,
    VALIDATION_PATH,
    TEST_PATH,
    TEXT_COLUMN,
    TARGET_COLUMN,
    MODELS_DIR,
)

from src.pipelines import (
    build_logistic_pipeline,
    build_svm_pipeline,
)

from src.evaluation import (
    calculate_classification_metrics,
    create_results_table,
)

1. Завантажуємо дані

In [3]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VALIDATION_PATH)
test_df = pd.read_csv(TEST_PATH)

X_train = train_df[TEXT_COLUMN]
y_train = train_df[TARGET_COLUMN]

X_val = val_df[TEXT_COLUMN]
y_val = val_df[TARGET_COLUMN]

X_test = test_df[TEXT_COLUMN]
y_test = test_df[TARGET_COLUMN]

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (7999,)
Validation: (2000,)
Test: (3080,)


2. Робимо Dummy baseline

In [5]:
results = []

dummy_model = DummyClassifier(
    strategy="most_frequent",
    random_state=42,
)

start_time = time.perf_counter()

dummy_model.fit(X_train, y_train)

dummy_train_time = time.perf_counter() - start_time
dummy_val_pred = dummy_model.predict(X_val)

dummy_result = calculate_classification_metrics(
    model_name="DummyClassifier",
    y_true=y_val,
    y_pred=dummy_val_pred,
    train_time=dummy_train_time,
    parameters={
        "strategy": "most_frequent",
    },
)

dummy_result.update({
    "variant": "baseline",
    "split": "validation",
    "n_features": None,
})

results_df = save_experiment_result(
    dummy_result,
    MODEL_RESULTS_PATH,
)

results.append(dummy_result)

create_results_table(results)

,model,parameters,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,train_time_sec,variant,split,n_features
0,DummyClassifier,{'strategy': 'most_frequent'},0.019,0.000484,0.000709,0.000247,0.012987,0.001041,baseline,validation,None


Як бачимо з результатів, DummyClassifier показує дуже низькі результати, оскільки він завжди передбачає найчастіший клас. Це підтверджує, що завдання не може бути вирішене за допомогою наївних стратегій і вимагає семантичного розуміння запитів клієнтів.

3. Тренуємо Logistic Regression

In [7]:
logistic_pipeline = build_logistic_pipeline()

In [8]:
start_time = time.perf_counter()

logistic_pipeline.fit(X_train, y_train)

logistic_train_time = time.perf_counter() - start_time
logistic_val_pred = logistic_pipeline.predict(X_val)

In [9]:
logistic_result = calculate_classification_metrics(
    model_name="TF-IDF + Logistic Regression",
    y_true=y_val,
    y_pred=logistic_val_pred,
    train_time=logistic_train_time,
    parameters={
        "ngram_range": (1, 2),
        "min_df": 2,
        "sublinear_tf": True,
        "C": 1.0,
    },
)

logistic_result.update({
    "variant": "baseline",
    "split": "validation",
    "n_features": len(
        logistic_pipeline
        .named_steps["tfidf"]
        .get_feature_names_out()
    ),
})

results_df = save_experiment_result(
    logistic_result,
    MODEL_RESULTS_PATH,
)

results.append(logistic_result)

create_results_table(results)

,model,parameters,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,train_time_sec,variant,split,n_features
0,TF-IDF + Logistic Regression,"{'ngram_range': (1, 2), 'min_df': 2, 'sublinea...",0.841,0.829135,0.839451,0.854706,0.824639,2.555273,baseline,validation,8913.0
1,DummyClassifier,{'strategy': 'most_frequent'},0.019,0.000484,0.000709,0.000247,0.012987,0.001041,baseline,validation,NaN


In [10]:
tfidf = logistic_pipeline.named_steps["tfidf"]

print(
    "Number of TF-IDF features:",
    len(tfidf.get_feature_names_out()),
)

Number of TF-IDF features: 8913


In [11]:
logistic_result["n_features"] = len(
    tfidf.get_feature_names_out()
)

In [14]:
label_mapping = (
    train_df[["label", "label_text"]]
    .drop_duplicates()
    .sort_values("label")
    .set_index("label")["label_text"]
    .to_dict()
)

target_names = [
    label_mapping[label]
    for label in sorted(label_mapping)
]

In [13]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_val,
        logistic_val_pred,
        labels=sorted(label_mapping),
        target_names=target_names,
        zero_division=0,
    )
)

                                                  precision    recall  f1-score   support

                                activate_my_card       0.97      0.91      0.94        32
                                       age_limit       0.95      0.95      0.95        22
                         apple_pay_or_google_pay       1.00      1.00      1.00        25
                                     atm_support       0.83      0.88      0.86        17
                                automatic_top_up       1.00      0.84      0.91        25
         balance_not_updated_after_bank_transfer       0.75      0.79      0.77        34
balance_not_updated_after_cheque_or_cash_deposit       0.86      1.00      0.92        36
                         beneficiary_not_allowed       0.92      0.77      0.84        31
                                 cancel_transfer       0.93      0.90      0.92        31
                            card_about_to_expire       0.80      0.92      0.86        26
         

4. Робимо Linear SVM

In [4]:
svm_pipeline = build_svm_pipeline()

In [5]:
start_time = time.perf_counter()

svm_pipeline.fit(X_train, y_train)

svm_train_time = time.perf_counter() - start_time
svm_val_pred = svm_pipeline.predict(X_val)

In [16]:
svm_result = calculate_classification_metrics(
    model_name="TF-IDF + Linear SVM",
    y_true=y_val,
    y_pred=svm_val_pred,
    train_time=svm_train_time,
    parameters={
        "ngram_range": (1, 2),
        "min_df": 2,
        "sublinear_tf": True,
        "C": 1.0,
    },
)

svm_result.update({
    "variant": "baseline",
    "split": "validation",
    "n_features": len(
        svm_pipeline
        .named_steps["tfidf"]
        .get_feature_names_out()
    ),
})

results_df = save_experiment_result(
    svm_result,
    MODEL_RESULTS_PATH,
)

svm_tfidf = svm_pipeline.named_steps["tfidf"]

svm_result["n_features"] = len(
    svm_tfidf.get_feature_names_out()
)

results.append(svm_result)

create_results_table(results)

,model,parameters,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,train_time_sec,variant,split,n_features
0,TF-IDF + Linear SVM,"{'ngram_range': (1, 2), 'min_df': 2, 'sublinea...",0.8755,0.870909,0.875092,0.879338,0.868093,0.367386,baseline,validation,8913.0
1,TF-IDF + Logistic Regression,"{'ngram_range': (1, 2), 'min_df': 2, 'sublinea...",0.8410,0.829135,0.839451,0.854706,0.824639,2.555273,baseline,validation,8913.0
2,DummyClassifier,{'strategy': 'most_frequent'},0.0190,0.000484,0.000709,0.000247,0.012987,0.001041,baseline,validation,NaN


5. Порівняємо результати

In [18]:
results_df = create_results_table(results)

results_df[
    [
        "model",
        "accuracy",
        "macro_f1",
        "weighted_f1",
        "macro_precision",
        "macro_recall",
        "train_time_sec",
        "n_features",
    ]
]

,model,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,train_time_sec,n_features
0,TF-IDF + Linear SVM,0.8755,0.870909,0.875092,0.879338,0.868093,0.367386,8913.0
1,TF-IDF + Logistic Regression,0.8410,0.829135,0.839451,0.854706,0.824639,2.555273,8913.0
2,DummyClassifier,0.0190,0.000484,0.000709,0.000247,0.012987,0.001041,NaN


Як бачимо із порівняльної таблиці, моделі TF-IDF + Linier SVM та TF-IDF + Logistic Regression показуюють дуже хороші результати навіть без тюнінгу. Macro F1 Linear SVM тановить 0.87, а для Logistic Regression 0.83.

6. Робимо тюнінг Logistic Regression і Linear SVM

6.1 GridSearch для Logistic Regression

In [19]:
logistic_pipeline = build_logistic_pipeline()

logistic_param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2],
    "classifier__C": [0.5, 1.0, 2.0],
}

logistic_grid = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

In [20]:
import time
import warnings
warnings.filterwarnings('ignore')

start_time = time.perf_counter()

logistic_grid.fit(X_train, y_train)

logistic_grid_time = time.perf_counter() - start_time

Fitting 3 folds for each of 12 candidates, totalling 36 fits


In [21]:
print("Best parameters:")
print(logistic_grid.best_params_)

print("Best CV Macro F1:")
print(logistic_grid.best_score_)

Best parameters:
{'classifier__C': 2.0, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 1)}
Best CV Macro F1:
0.8529892233070906


Оцінка на validation

In [22]:
logistic_tuned_val_pred = logistic_grid.predict(X_val)

logistic_tuned_result = calculate_classification_metrics(
    model_name="TF-IDF + Logistic Regression",
    y_true=y_val,
    y_pred=logistic_tuned_val_pred,
    train_time=logistic_grid_time,
    parameters=logistic_grid.best_params_,
)

logistic_tuned_result.update({
    "variant": "tuned",
    "split": "validation",
    "n_features": len(
        logistic_grid
        .best_estimator_
        .named_steps["tfidf"]
        .get_feature_names_out()
    ),
})

save_experiment_result(
    logistic_tuned_result,
    MODEL_RESULTS_PATH,
)

logistic_tuned_result["n_features"] = len(
    logistic_grid
    .best_estimator_
    .named_steps["tfidf"]
    .get_feature_names_out()
)

results.append(logistic_tuned_result)

create_results_table(results)

,model,parameters,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,train_time_sec,variant,split,n_features
0,TF-IDF + Linear SVM,"{'ngram_range': (1, 2), 'min_df': 2, 'sublinea...",0.8755,0.870909,0.875092,0.879338,0.868093,0.367386,baseline,validation,8913.0
1,TF-IDF + Logistic Regression,"{'classifier__C': 2.0, 'tfidf__min_df': 2, 'tf...",0.8675,0.862007,0.867527,0.874387,0.858843,15.551374,tuned,validation,1331.0
2,TF-IDF + Logistic Regression,"{'ngram_range': (1, 2), 'min_df': 2, 'sublinea...",0.8410,0.829135,0.839451,0.854706,0.824639,2.555273,baseline,validation,8913.0
3,DummyClassifier,{'strategy': 'most_frequent'},0.0190,0.000484,0.000709,0.000247,0.012987,0.001041,baseline,validation,NaN


6.2. GridSearch для Linear SVM

In [9]:
svm_pipeline = build_svm_pipeline()

svm_param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2],
    "classifier__C": [0.5, 1.0, 2.0],
}

svm_grid = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

In [10]:
start_time = time.perf_counter()

svm_grid.fit(X_train, y_train)

svm_grid_time = time.perf_counter() - start_time

Fitting 3 folds for each of 12 candidates, totalling 36 fits


In [25]:
print("Best parameters:")
print(svm_grid.best_params_)

print("Best CV Macro F1:")
print(svm_grid.best_score_)

Best parameters:
{'classifier__C': 1.0, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 1)}
Best CV Macro F1:
0.8638637953920215


In [26]:
svm_tuned_val_pred = svm_grid.predict(X_val)

svm_tuned_result = calculate_classification_metrics(
    model_name="TF-IDF + Linear SVM",
    y_true=y_val,
    y_pred=svm_tuned_val_pred,
    train_time=svm_grid_time,
    parameters=svm_grid.best_params_,
)

svm_tuned_result.update({
    "variant": "tuned",
    "split": "validation",
    "n_features": len(
        svm_grid
        .best_estimator_
        .named_steps["tfidf"]
        .get_feature_names_out()
    ),
})

save_experiment_result(
    svm_tuned_result,
    MODEL_RESULTS_PATH,
)

svm_tuned_result["n_features"] = len(
    svm_grid
    .best_estimator_
    .named_steps["tfidf"]
    .get_feature_names_out()
)

results.append(svm_tuned_result)

results_df = create_results_table(results)
results_df

,model,parameters,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,train_time_sec,variant,split,n_features
0,TF-IDF + Linear SVM,"{'classifier__C': 1.0, 'tfidf__min_df': 1, 'tf...",0.8790,0.873237,0.878779,0.881031,0.871352,3.870460,tuned,validation,2158.0
1,TF-IDF + Linear SVM,"{'ngram_range': (1, 2), 'min_df': 2, 'sublinea...",0.8755,0.870909,0.875092,0.879338,0.868093,0.367386,baseline,validation,8913.0
2,TF-IDF + Logistic Regression,"{'classifier__C': 2.0, 'tfidf__min_df': 2, 'tf...",0.8675,0.862007,0.867527,0.874387,0.858843,15.551374,tuned,validation,1331.0
3,TF-IDF + Logistic Regression,"{'ngram_range': (1, 2), 'min_df': 2, 'sublinea...",0.8410,0.829135,0.839451,0.854706,0.824639,2.555273,baseline,validation,8913.0
4,DummyClassifier,{'strategy': 'most_frequent'},0.0190,0.000484,0.000709,0.000247,0.012987,0.001041,baseline,validation,NaN


7. Зберігаємо найкращі класичні моделі

In [16]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    logistic_grid.best_estimator_,
    MODELS_DIR / "logistic_pipeline_tuned.joblib",
)

joblib.dump(
    svm_grid.best_estimator_,
    MODELS_DIR / "svm_pipeline_tuned.joblib",
    compress=("xz", 3),
)

['/Users/oleh/PycharmProjects/banking-intent-classification/models/svm_pipeline_tuned.joblib']

7.1 Тренуємо і зберігаємо модель для деплою (робимо CalibratedClassifierCV щоб мати можливість отримувати Probability)

In [21]:
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline

best_pipeline = svm_grid.best_estimator_

# створюємо новий пайплайн
deployment_pipeline = Pipeline([
    (
        "tfidf",
        clone(best_pipeline.named_steps["tfidf"]),
    ),

    (
        "classifier",
        CalibratedClassifierCV(
            estimator=clone(
                best_pipeline.named_steps["classifier"]
            ),
            method="sigmoid",
            cv=5,
            n_jobs=-1,
        ),
    ),
])

# тренуємо модель
deployment_pipeline.fit(
    X_train,
    y_train,
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](77,)","[ 0, 1, 2,...,74,75,76]"
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",False
,"sublinear_tf sublinear_tf: bool, default=FalseApply sublinear tf scaling, i.e. replace tf with 1 + log(tf).",True
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


In [22]:
import joblib

label_mapping = (
    train_df[["label", "label_text"]]
    .drop_duplicates()
    .sort_values("label")
    .set_index("label")["label_text"]
    .to_dict()
)

deployment_model = {
    "pipeline": deployment_pipeline,
    "label_mapping": label_mapping,
}

joblib.dump(
    deployment_model,
    MODELS_DIR / "svm_deployment.joblib",
    compress=("xz", 3),
)

['/Users/oleh/PycharmProjects/banking-intent-classification/models/svm_deployment.joblib']

In [24]:
import numpy as np

text = "How can I activate my card?"

probabilities = deployment_pipeline.predict_proba([text])[0]
classes = deployment_pipeline.classes_

top_indices = np.argsort(probabilities)[::-1][:3]

for index in top_indices:
    label = int(classes[index])

    print(
        label_mapping[label],
        f"{probabilities[index]:.2%}",
    )

activate_my_card 93.48%
card_acceptance 0.65%
lost_or_stolen_card 0.50%


8. Оцінюємо на test

In [27]:
test_df = pd.read_csv(TEST_PATH)

X_test = test_df[TEXT_COLUMN]
y_test = test_df[TARGET_COLUMN]

print("Test size:", len(test_df))

Test size: 3080


8.1 Оцінюємо Logistic Regression на test

In [33]:
logistic_model_path = (
    MODELS_DIR / "logistic_pipeline_tuned.joblib"
)

logistic_model = joblib.load(logistic_model_path)

start_time = time.perf_counter()
logistic_test_pred = logistic_model.predict(X_test)
logistic_inference_time = time.perf_counter() - start_time

In [34]:
parameters={
    "ngram_range": logistic_model.named_steps["tfidf"].ngram_range,
    "min_df": logistic_model.named_steps["tfidf"].min_df,
    "sublinear_tf": logistic_model.named_steps["tfidf"].sublinear_tf,
    "C": logistic_model.named_steps["classifier"].C,
}

logistic_test_result = calculate_classification_metrics(
    model_name="TF-IDF + Logistic Regression",
    y_true=y_test,
    y_pred=logistic_test_pred,
    parameters=parameters,
)

logistic_test_result.update({
    "variant": "tuned",
    "split": "test",
    "inference_time_sec": logistic_inference_time,
    "n_features": len(
        logistic_model
        .named_steps["tfidf"]
        .get_feature_names_out()
    ),
})

save_experiment_result(
    logistic_test_result,
    MODEL_RESULTS_PATH,
)

,model,parameters,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,train_time_sec,variant,split,n_features,inference_time_sec
0,DistilBERT,"{""checkpoint"": ""distilbert-base-uncased"", ""epo...",0.909416,0.909147,0.909147,0.914840,0.909416,NaN,fine-tuned,test,NaN,5.714908
1,DistilBERT,"{""checkpoint"": ""distilbert-base-uncased"", ""epo...",0.906500,0.900707,0.905698,0.913445,0.898439,420.774839,fine-tuned,validation,NaN,NaN
2,TF-IDF + Logistic Regression,"{""C"": 2.0, ""min_df"": 2, ""ngram_range"": [1, 1],...",0.881818,0.881908,0.881908,0.887893,0.881818,None,tuned,test,1331.0,0.016299
3,TF-IDF + Linear SVM,"{""classifier__C"": 1.0, ""tfidf__min_df"": 1, ""tf...",0.879000,0.873237,0.878779,0.881031,0.871352,3.87046,tuned,validation,2158.0,NaN
4,TF-IDF + Linear SVM,"{""C"": 1.0, ""min_df"": 2, ""ngram_range"": [1, 2],...",0.875500,0.870909,0.875092,0.879338,0.868093,0.367386,baseline,validation,8913.0,NaN
5,TF-IDF + Logistic Regression,"{""classifier__C"": 2.0, ""tfidf__min_df"": 2, ""tf...",0.867500,0.862007,0.867527,0.874387,0.858843,15.551374,tuned,validation,1331.0,NaN
6,TF-IDF + Logistic Regression,"{""C"": 1.0, ""min_df"": 2, ""ngram_range"": [1, 2],...",0.841000,0.829135,0.839451,0.854706,0.824639,2.555273,baseline,validation,8913.0,NaN
7,DummyClassifier,"{""strategy"": ""most_frequent""}",0.019000,0.000484,0.000709,0.000247,0.012987,0.001041,baseline,validation,NaN,NaN


8.2 Оцінюємо Linear SVM на test

In [35]:
svm_model_path = MODELS_DIR / "svm_pipeline_tuned.joblib"

svm_model = joblib.load(svm_model_path)

start_time = time.perf_counter()
svm_test_pred = svm_model.predict(X_test)
svm_inference_time = time.perf_counter() - start_time

In [36]:
svm_test_result = calculate_classification_metrics(
    model_name="TF-IDF + Linear SVM",
    y_true=y_test,
    y_pred=svm_test_pred,
    parameters={
        "ngram_range": svm_model
            .named_steps["tfidf"]
            .ngram_range,
        "min_df": svm_model
            .named_steps["tfidf"]
            .min_df,
        "C": svm_model
            .named_steps["classifier"]
            .C,
    },
)

svm_test_result.update({
    "variant": "tuned",
    "split": "test",
    "inference_time_sec": svm_inference_time,
    "n_features": len(
        svm_model
        .named_steps["tfidf"]
        .get_feature_names_out()
    ),
})

save_experiment_result(
    svm_test_result,
    MODEL_RESULTS_PATH,
)

,model,parameters,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,train_time_sec,variant,split,n_features,inference_time_sec
0,DistilBERT,"{""checkpoint"": ""distilbert-base-uncased"", ""epo...",0.909416,0.909147,0.909147,0.914840,0.909416,NaN,fine-tuned,test,NaN,5.714908
1,DistilBERT,"{""checkpoint"": ""distilbert-base-uncased"", ""epo...",0.906500,0.900707,0.905698,0.913445,0.898439,420.774839,fine-tuned,validation,NaN,NaN
2,TF-IDF + Linear SVM,"{""C"": 1.0, ""min_df"": 1, ""ngram_range"": [1, 1]}",0.887987,0.887864,0.887864,0.892381,0.887987,None,tuned,test,2158.0,0.021133
3,TF-IDF + Logistic Regression,"{""C"": 2.0, ""min_df"": 2, ""ngram_range"": [1, 1],...",0.881818,0.881908,0.881908,0.887893,0.881818,NaN,tuned,test,1331.0,0.016299
4,TF-IDF + Linear SVM,"{""classifier__C"": 1.0, ""tfidf__min_df"": 1, ""tf...",0.879000,0.873237,0.878779,0.881031,0.871352,3.87046,tuned,validation,2158.0,NaN
5,TF-IDF + Linear SVM,"{""C"": 1.0, ""min_df"": 2, ""ngram_range"": [1, 2],...",0.875500,0.870909,0.875092,0.879338,0.868093,0.367386,baseline,validation,8913.0,NaN
6,TF-IDF + Logistic Regression,"{""classifier__C"": 2.0, ""tfidf__min_df"": 2, ""tf...",0.867500,0.862007,0.867527,0.874387,0.858843,15.551374,tuned,validation,1331.0,NaN
7,TF-IDF + Logistic Regression,"{""C"": 1.0, ""min_df"": 2, ""ngram_range"": [1, 2],...",0.841000,0.829135,0.839451,0.854706,0.824639,2.555273,baseline,validation,8913.0,NaN
8,DummyClassifier,"{""strategy"": ""most_frequent""}",0.019000,0.000484,0.000709,0.000247,0.012987,0.001041,baseline,validation,NaN,NaN


8.3 Оцінюємо DummyClassifier на test

In [37]:
dummy_test_pred = dummy_model.predict(X_test)

dummy_test_result = calculate_classification_metrics(
    model_name="DummyClassifier",
    y_true=y_test,
    y_pred=dummy_test_pred,
    train_time=dummy_train_time,
    parameters={
        "strategy": "most_frequent",
    },
)

dummy_test_result.update({
    "variant": "baseline",
    "split": "test",
    "n_features": None,
})

results_test_df = save_experiment_result(
    dummy_test_result,
    MODEL_RESULTS_PATH,
)